<a href="https://colab.research.google.com/github/harneet862/ur2phd-assignment/blob/main/part2_deberta_seed2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 2: System A, fine-tuned DeBERTa-v3-base (seed 2026)
This notebook is identical for every seed. Only `SEED` changes. Each seed runs in its own notebook so a fresh session always starts from a fresh model.

## Setup

In [2]:
!pip install -q transformers datasets accelerate sentencepiece scikit-learn
!git clone https://github.com/ilopezgazpio/SemEval-2027-Task-2-DiCo-NLI.git

fatal: destination path 'SemEval-2027-Task-2-DiCo-NLI' already exists and is not an empty directory.


In [3]:
import pandas as pd, os

TASK_DIR = "/content/SemEval-2027-Task-2-DiCo-NLI"
TRAIN_PATH = f"{TASK_DIR}/final_data/train/dico_nli_train_track1_participant_labeled.csv"
DEV_REF_PATH = f"{TASK_DIR}/final_data/dev/dico_nli_dev_track1_reference.csv"
RESULTS_DIR = f"{TASK_DIR}/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

In [4]:
SEED = 2026

## Labels

In [5]:
LABELS = ["FORWARD_ENTAILMENT", "BACKWARD_ENTAILMENT", "EQUIVALENCE", "NEGATIVE_OTHER"]

label2id = dict()
id2label = dict()
for idx, elem in enumerate(LABELS):
    label2id[elem] = idx
    id2label[idx] = elem
print(label2id)

{'FORWARD_ENTAILMENT': 0, 'BACKWARD_ENTAILMENT': 1, 'EQUIVALENCE': 2, 'NEGATIVE_OTHER': 3}


## Data: train and dev

In [6]:
from datasets import Dataset

train = pd.read_csv(TRAIN_PATH)
train["labels"] = train["label"].map(label2id)
print("Train rows:", len(train), "| NaN labels:", train["labels"].isna().sum())

dev = pd.read_csv(DEV_REF_PATH)
dev["labels"] = dev["label"].map(label2id)
print("Dev rows:", len(dev), "| NaN labels:", dev["labels"].isna().sum())

Train rows: 3042 | NaN labels: 0
Dev rows: 660 | NaN labels: 0


In [7]:
from transformers import AutoTokenizer

MODEL_NAME = "microsoft/deberta-v3-base"
tokeniser = AutoTokenizer.from_pretrained(MODEL_NAME)

# both phrases go in as one sequence: [CLS] text1 [SEP] text2 [SEP]
def tokenize_pairs(batch):
    return tokeniser(batch["text1"], batch["text2"])

train_dataset = Dataset.from_pandas(train[["text1", "text2", "labels"]])
tokenized_train = train_dataset.map(tokenize_pairs, batched=True)

dev_dataset = Dataset.from_pandas(dev[["text1", "text2", "labels"]])
tokenized_dev = dev_dataset.map(tokenize_pairs, batched=True)

print(tokenized_train)
print(tokenized_dev)

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

Map:   0%|          | 0/3042 [00:00<?, ? examples/s]

Map:   0%|          | 0/660 [00:00<?, ? examples/s]

Dataset({
    features: ['text1', 'text2', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 3042
})
Dataset({
    features: ['text1', 'text2', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 660
})


## Model (fresh for this seed)

In [8]:
import torch
from transformers import set_seed, AutoModelForSequenceClassification

set_seed(SEED)

# weights loaded in float32; fp16 mixed precision is handled by the Trainer
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    label2id=label2id,
    id2label=id2label,
    torch_dtype=torch.float32,
)
print(next(model.parameters()).dtype)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  371MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.den

torch.float32


## Trainer
Settings are fixed for all seeds: lr 2e-5, batch 16, 5 epochs, weight decay 0.01, no warmup, fp16, final checkpoint, no evaluation during training.

In [9]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

training_args = TrainingArguments(
    output_dir="deberta_out",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    fp16=True,
    seed=SEED,
    save_strategy="no",
    logging_steps=50,
    report_to="none",
)

collator = DataCollatorWithPadding(tokenizer=tokeniser)  # pads each batch to its longest item

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=collator,
    processing_class=tokeniser,
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  371MB            

model.safetensors: downloading bytes:           |  0.00B            

## Train
Expected: 955/955 steps, loss near 1.35 at step 50.

In [10]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Step,Training Loss
50,1.358491
100,1.179946
150,0.875250
200,0.696822
250,0.532906
300,0.447052
350,0.418328
400,0.352760
450,0.243022
500,0.197188


TrainOutput(global_step=955, training_loss=0.3875480372244151, metrics={'train_runtime': 119.5641, 'train_samples_per_second': 127.212, 'train_steps_per_second': 7.987, 'total_flos': 119270671989264.0, 'train_loss': 0.3875480372244151, 'epoch': 5.0})

In [11]:
# save the training loss log for this seed
loss_df = pd.DataFrame(trainer.state.log_history)
loss_df.to_csv(f"{RESULTS_DIR}/train_loss_seed{SEED}.csv", index=False)
print(loss_df[["step", "loss"]].dropna().to_string(index=False))

 step     loss
   50 1.358491
  100 1.179946
  150 0.875250
  200 0.696822
  250 0.532906
  300 0.447052
  350 0.418328
  400 0.352760
  450 0.243022
  500 0.197188
  550 0.240477
  600 0.174144
  650 0.114631
  700 0.149395
  750 0.124404
  800 0.082167
  850 0.069334
  900 0.055447
  950 0.085916


## Predict on dev and save

In [12]:
out = trainer.predict(tokenized_dev)
pred_ids = out.predictions.argmax(-1)
pred_labels = [id2label[int(i)] for i in pred_ids]

print(len(pred_labels))
print(pd.Series(pred_labels).value_counts())

660
FORWARD_ENTAILMENT     187
BACKWARD_ENTAILMENT    184
EQUIVALENCE            176
NEGATIVE_OTHER         113
Name: count, dtype: int64


In [13]:
# rows keep the same order as dev, so position i in pred_labels belongs to dev row i
preds_df = pd.DataFrame({"instance_id": dev["instance_id"], "label": pred_labels})
PRED_PATH = f"{RESULTS_DIR}/predictions_deberta_seed{SEED}.csv"
preds_df.to_csv(PRED_PATH, index=False)
print(PRED_PATH)
print(preds_df.head())

/content/SemEval-2027-Task-2-DiCo-NLI/results/predictions_deberta_seed2026.csv
                             instance_id                label
0   dico_phrasis_0000002__en-en__flipped   FORWARD_ENTAILMENT
1  dico_phrasis_0000002__en-en__original  BACKWARD_ENTAILMENT
2   dico_phrasis_0000004__en-en__flipped   FORWARD_ENTAILMENT
3  dico_phrasis_0000004__en-en__original  BACKWARD_ENTAILMENT
4   dico_phrasis_0000008__en-en__flipped   FORWARD_ENTAILMENT


## Score with the official scorer

In [14]:
%cd /content/SemEval-2027-Task-2-DiCo-NLI
!python3 -m evaluation_functions \
  --gold final_data/dev/dico_nli_dev_track1_reference.csv \
  --predictions results/predictions_deberta_seed2026.csv \
  --output-dir results/deberta_seed2026

/content/SemEval-2027-Task-2-DiCo-NLI
{
  "classification": {
    "accuracy": 0.8136363636363636,
    "confusion_matrix": {
      "BACKWARD_ENTAILMENT": {
        "BACKWARD_ENTAILMENT": 165,
        "EQUIVALENCE": 10,
        "FORWARD_ENTAILMENT": 2,
        "NEGATIVE_OTHER": 13
      },
      "EQUIVALENCE": {
        "BACKWARD_ENTAILMENT": 6,
        "EQUIVALENCE": 138,
        "FORWARD_ENTAILMENT": 8,
        "NEGATIVE_OTHER": 22
      },
      "FORWARD_ENTAILMENT": {
        "BACKWARD_ENTAILMENT": 1,
        "EQUIVALENCE": 13,
        "FORWARD_ENTAILMENT": 166,
        "NEGATIVE_OTHER": 10
      },
      "NEGATIVE_OTHER": {
        "BACKWARD_ENTAILMENT": 12,
        "EQUIVALENCE": 15,
        "FORWARD_ENTAILMENT": 11,
        "NEGATIVE_OTHER": 68
      }
    },
    "macro_f1": 0.7931413851831202,
    "per_label": {
      "BACKWARD_ENTAILMENT": {
        "f1": 0.8823529411764707,
        "false_negative": 25,
        "false_positive": 19,
        "label": "BACKWARD_ENTAILMENT",
     

In [15]:
!cat results/deberta_seed2026/scores.txt

weighted_f1: 0.815160494425
soft_cons: 0.895306859206
hard_cons: 0.833935018051
